<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Flash-Attention/flash-attention.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("flash-attention", ...)`


## Flash Attention Forward Pass

FlashAttention computes exact softmax attention without ever materializing the `N_q × N_k` score matrix. It walks over K/V in tiles and keeps, for every query row, a running max `m`, a running softmax denominator `l` and an unnormalized output accumulator `acc`. For each K/V tile `j`, with scaled scores `S_j = Q K_jᵀ / sqrt(D)`:

```
m_new = max(m, rowmax(S_j))
P_j   = exp(S_j - m_new)
alpha = exp(m - m_new)              # rescales everything accumulated under the old max
l     = alpha * l   + rowsum(P_j)
acc   = alpha * acc + P_j @ V_j
m     = m_new
```

After the last tile, `O = acc / l` and `L = m + log(l)` (the logsumexp of each row of scaled scores, which the backward pass needs). Start from `m = -inf`, `l = 0`, `acc = 0`.

### Part 1 — PyTorch (graded, runs on CPU)

Implement `flash_attention_pytorch(Q, K, V, block_size_q=16, block_size_k=16)`:

- `Q`: `(B, N_q, D)`, `K`, `V`: `(B, N_k, D)`; return `(O, L)` with `O`: `(B, N_q, D)` and `L`: `(B, N_q)`.
- Loop over query tiles and K/V tiles with the recurrence above. Do not compute the full score matrix or call `softmax`.
- The result must equal `F.scaled_dot_product_attention(Q, K, V)` for **any** tile sizes.

### Part 2 — Triton forward kernel (needs a GPU, or the CPU interpreter)

Implement the same loop as a **Triton kernel** `flash_fwd_kernel` that writes `O` and `L`:

- Launch it on a grid of **`(T_q, batch_size)`**: each program instance handles **one tile of `Q`** for a **single batch index**.
- Within each program instance:
  - load the query tile, then stream the batch's K/V tiles through it,
  - apply the online-softmax recurrence above,
  - store the output tile in `O` and the logsumexp values in `L`.
- Address tiles with plain pointer arithmetic (`base + offsets[:, None] * stride_row + offsets[None, :] * stride_col`). `tl.make_block_ptr` / `tl.advance` were removed in Triton 3.9.
- Without a GPU you can run the kernel on CPU with `TRITON_INTERPRET=1` set before importing Triton (slow but exact).

### Notes

- We test with sequence lengths that are multiples of the block sizes (powers of 2, at least 16), so you don't need to worry about out-of-bounds accesses.

In [ ]:
import math
import os

import torch

# Triton kernels need a GPU, or the CPU interpreter (set TRITON_INTERPRET=1 before
# importing triton). Guard the import so the notebook still opens, and the PyTorch
# reference below still runs, on a machine without Triton.
CUDA_AVAILABLE = torch.cuda.is_available()
TRITON_INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
except ImportError:
    TRITON_AVAILABLE = False

print(f"CUDA available: {CUDA_AVAILABLE} | Triton available: {TRITON_AVAILABLE} | "
      f"interpreter: {TRITON_INTERPRET}")

if TRITON_AVAILABLE:
    @triton.jit
    def flash_fwd_kernel(Q_ptr, K_ptr, V_ptr,
                         O_ptr, L_ptr,
                         stride_qb, stride_qq, stride_qd,
                         stride_kb, stride_kk, stride_kd,
                         stride_vb, stride_vk, stride_vd,
                         stride_ob, stride_ok, stride_od,
                         stride_lb, stride_lq,
                         N_q, N_k,
                         scale,
                         D: tl.constexpr,
                         BLOCK_SIZE_Q: tl.constexpr,
                         BLOCK_SIZE_K: tl.constexpr):

        # Program indices: one query tile of one batch element
        query_tile_index = tl.program_id(0)
        batch_index = tl.program_id(1)

        # Row / column offsets of the tiles this program touches
        offs_q = query_tile_index * BLOCK_SIZE_Q + tl.arange(0, BLOCK_SIZE_Q)
        offs_k = tl.arange(0, BLOCK_SIZE_K)
        offs_d = tl.arange(0, D)

        # (BLOCK_SIZE_Q, D) query tile
        q_ptrs = (Q_ptr + batch_index * stride_qb
                  + offs_q[:, None] * stride_qq + offs_d[None, :] * stride_qd)
        ####### Your Code goes here ############

        pass

In [ ]:
# Part 1: the tiled forward pass in PyTorch (graded on CPU with
# check("flash-attention", flash_attention_pytorch)). Uncomment and fill in:
#
# def flash_attention_pytorch(Q, K, V, block_size_q=16, block_size_k=16):
#     """Q: (B, N_q, D), K/V: (B, N_k, D) -> O: (B, N_q, D), L: (B, N_q)."""
#     ...

## Solution

In [ ]:
import math
import os

import torch

# Triton kernels need a GPU, or the CPU interpreter (set TRITON_INTERPRET=1 before
# importing triton). Guard the import so the notebook still opens, and the PyTorch
# reference below still runs, on a machine without Triton.
CUDA_AVAILABLE = torch.cuda.is_available()
TRITON_INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
except ImportError:
    TRITON_AVAILABLE = False

print(f"CUDA available: {CUDA_AVAILABLE} | Triton available: {TRITON_AVAILABLE} | "
      f"interpreter: {TRITON_INTERPRET}")

if TRITON_AVAILABLE:
    @triton.jit
    def flash_fwd_kernel(Q_ptr, K_ptr, V_ptr,
                         O_ptr, L_ptr,
                         stride_qb, stride_qq, stride_qd,
                         stride_kb, stride_kk, stride_kd,
                         stride_vb, stride_vk, stride_vd,
                         stride_ob, stride_ok, stride_od,
                         stride_lb, stride_lq,
                         N_q, N_k,
                         scale,
                         D: tl.constexpr,
                         BLOCK_SIZE_Q: tl.constexpr,
                         BLOCK_SIZE_K: tl.constexpr):

        # Program indices: one query tile of one batch element
        query_tile_index = tl.program_id(0)
        batch_index = tl.program_id(1)

        # Row / column offsets of the tiles this program touches
        offs_q = query_tile_index * BLOCK_SIZE_Q + tl.arange(0, BLOCK_SIZE_Q)
        offs_k = tl.arange(0, BLOCK_SIZE_K)
        offs_d = tl.arange(0, D)

        # (BLOCK_SIZE_Q, D) query tile
        q_ptrs = (Q_ptr + batch_index * stride_qb
                  + offs_q[:, None] * stride_qq + offs_d[None, :] * stride_qd)
        # K is read transposed, as a (D, BLOCK_SIZE_K) tile; V as (BLOCK_SIZE_K, D)
        k_ptrs = (K_ptr + batch_index * stride_kb
                  + offs_k[None, :] * stride_kk + offs_d[:, None] * stride_kd)
        v_ptrs = (V_ptr + batch_index * stride_vb
                  + offs_k[:, None] * stride_vk + offs_d[None, :] * stride_vd)

        q = tl.load(q_ptrs).to(tl.float32)

        # Online-softmax state for each query row
        m_i = tl.full([BLOCK_SIZE_Q], float("-inf"), dtype=tl.float32)  # running max
        l_i = tl.zeros([BLOCK_SIZE_Q], dtype=tl.float32)                # running denominator
        acc = tl.zeros([BLOCK_SIZE_Q, D], dtype=tl.float32)             # unnormalized output

        for start_k in range(0, N_k, BLOCK_SIZE_K):
            k = tl.load(k_ptrs + start_k * stride_kk).to(tl.float32)
            v = tl.load(v_ptrs + start_k * stride_vk).to(tl.float32)

            s = tl.dot(q, k) * scale                      # (BLOCK_SIZE_Q, BLOCK_SIZE_K)
            m_new = tl.maximum(m_i, tl.max(s, axis=1))
            p = tl.exp(s - m_new[:, None])

            # Rescale what was accumulated under the old max, then add this tile
            alpha = tl.exp(m_i - m_new)
            l_i = l_i * alpha + tl.sum(p, axis=1)
            acc = acc * alpha[:, None] + tl.dot(p, v)
            m_i = m_new

        # Normalize once, at the end
        acc = acc / l_i[:, None]
        o_ptrs = (O_ptr + batch_index * stride_ob
                  + offs_q[:, None] * stride_ok + offs_d[None, :] * stride_od)
        tl.store(o_ptrs, acc.to(O_ptr.dtype.element_ty))

        # Logsumexp of each row of scaled scores, kept for the backward pass
        l_ptrs = L_ptr + batch_index * stride_lb + offs_q * stride_lq
        tl.store(l_ptrs, (m_i + tl.log(l_i)).to(L_ptr.dtype.element_ty))

### PyTorch reference (runs without a GPU)


In [ ]:
def flash_attention_pytorch(Q, K, V, block_size_q=16, block_size_k=16):
    """Tiled ("flash") attention in pure PyTorch - runs anywhere.

    This is the algorithm the Triton kernel implements: walk the keys in blocks,
    keeping a running max and a running sum so the softmax never materialises the
    full N_q x N_k score matrix. Getting the rescale by exp(prev_max - new_max)
    right is the whole exercise.

    Args:
        Q: (B, N_q, D)
        K: (B, N_k, D)
        V: (B, N_k, D)

    Returns:
        O: (B, N_q, D) attention output
        L: (B, N_q) log-sum-exp of the scores, as the backward pass needs
    """
    B, N_q, D = Q.shape
    N_k = K.shape[1]
    scale = 1.0 / math.sqrt(D)

    O = torch.zeros(B, N_q, D, dtype=torch.float32, device=Q.device)
    L = torch.zeros(B, N_q, dtype=torch.float32, device=Q.device)

    Qf, Kf, Vf = Q.float(), K.float(), V.float()

    for q0 in range(0, N_q, block_size_q):
        q1 = min(q0 + block_size_q, N_q)
        q_blk = Qf[:, q0:q1, :]

        running_max = torch.full((B, q1 - q0), float("-inf"), device=Q.device)
        running_sum = torch.zeros(B, q1 - q0, device=Q.device)
        acc = torch.zeros(B, q1 - q0, D, device=Q.device)

        for k0 in range(0, N_k, block_size_k):
            k1 = min(k0 + block_size_k, N_k)
            scores = torch.matmul(q_blk, Kf[:, k0:k1, :].transpose(-2, -1)) * scale

            blk_max = scores.max(dim=-1).values
            new_max = torch.maximum(running_max, blk_max)

            # Rescale what we accumulated so far to the new maximum.
            correction = torch.exp(running_max - new_max)
            correction = torch.nan_to_num(correction, nan=0.0)

            p = torch.exp(scores - new_max.unsqueeze(-1))
            running_sum = running_sum * correction + p.sum(dim=-1)
            acc = acc * correction.unsqueeze(-1) + torch.matmul(p, Vf[:, k0:k1, :])
            running_max = new_max

        O[:, q0:q1, :] = acc / running_sum.unsqueeze(-1)
        L[:, q0:q1] = running_max + torch.log(running_sum)

    return O, L


# Quick check against PyTorch's own attention.
_B, _Nq, _Nk, _D = 2, 40, 48, 16
torch.manual_seed(0)
_Q = torch.randn(_B, _Nq, _D)
_K = torch.randn(_B, _Nk, _D)
_V = torch.randn(_B, _Nk, _D)

_O, _L = flash_attention_pytorch(_Q, _K, _V)
_expected = torch.nn.functional.scaled_dot_product_attention(_Q, _K, _V)
print("matches torch attention:", torch.allclose(_O, _expected, atol=1e-5))


## TESTS

In [ ]:
# Runs on a CUDA GPU, or on CPU when TRITON_INTERPRET=1 was set before importing triton.
if TRITON_AVAILABLE and (CUDA_AVAILABLE or TRITON_INTERPRET):
    device = "cuda" if CUDA_AVAILABLE else "cpu"
    dtype = torch.float16 if CUDA_AVAILABLE else torch.float32
    B, N_q, N_k, D = 1, 64, 128, 64
    BLOCK_SIZE_Q = 16
    BLOCK_SIZE_K = 16

    Q = torch.randn((B, N_q, D), dtype=dtype, device=device)
    K = torch.randn((B, N_k, D), dtype=dtype, device=device)
    V = torch.randn((B, N_k, D), dtype=dtype, device=device)

    O = torch.empty((B, N_q, D), dtype=dtype, device=device)
    L = torch.empty((B, N_q), dtype=torch.float32, device=device)

    grid = (triton.cdiv(N_q, BLOCK_SIZE_Q), B)

    flash_fwd_kernel[grid](
        Q, K, V, O, L,
        *Q.stride(), *K.stride(), *V.stride(), *O.stride(), *L.stride(),
        N_q, N_k,
        scale=1.0 / math.sqrt(D),
        D=D,
        BLOCK_SIZE_Q=BLOCK_SIZE_Q,
        BLOCK_SIZE_K=BLOCK_SIZE_K,
    )

    scale = 1.0 / math.sqrt(D)
    scores = torch.matmul(Q.float(), K.float().transpose(-2, -1)) * scale
    O_ref = torch.matmul(torch.softmax(scores, dim=-1), V.float())
    L_ref = torch.logsumexp(scores, dim=-1)

    print("O matches:", torch.allclose(O.float(), O_ref, atol=2e-2, rtol=1e-2))
    print("L matches:", torch.allclose(L.float(), L_ref, atol=1e-2, rtol=1e-2))
else:
    print("Skipping the Triton kernel test - it needs Triton plus a CUDA GPU or TRITON_INTERPRET=1.")